In [1]:
# get student no

import os
import re
import pandas as pd

# === CONFIGURATION ===
root_folder = r"YOUR_OUTPUT_FOLDER"   # change this
output_file = r"all student.xlsx"
error_log = "errors.txt"

# regex to find 11-digit sequences anywhere in the cell
student_no_pattern = re.compile(r"\b\d{11}\b")

all_student_numbers = []

# clear previous error log
with open(error_log, "w", encoding="utf-8") as f:
    f.write("=== Error Log ===\n")

# --- Step 1: collect all Excel files ---
excel_files = []
for folderpath, _, filenames in os.walk(root_folder):
    for filename in filenames:
        if filename.lower().endswith((".xlsx", ".xls")) and not filename.startswith("~$"):
            excel_files.append(os.path.join(folderpath, filename))

total_files = len(excel_files)
print(f"Found {total_files} Excel files to process.")

# --- Step 2: process files ---
for idx, filepath in enumerate(excel_files, start=1):
    try:
        # select engine
        if filepath.lower().endswith(".xls"):
            xls = pd.ExcelFile(filepath, engine="xlrd")
        else:
            xls = pd.ExcelFile(filepath, engine="openpyxl")

        for sheet in xls.sheet_names:
            df = pd.read_excel(filepath, sheet_name=sheet, dtype=str, engine=xls.engine)
            for col in df.columns:
                for value in df[col].dropna():
                    try:
                        value_str = str(value)
                        # remove hidden/non-breaking spaces
                        value_str = value_str.replace("\xa0", "").replace(" ", "").strip()
                        # find all 11-digit sequences
                        matches = student_no_pattern.findall(value_str)
                        for match in matches:
                            all_student_numbers.append([match, os.path.basename(filepath), sheet])
                    except Exception as e:
                        with open(error_log, "a", encoding="utf-8") as f:
                            f.write(f"Error processing value '{value}' in {filepath} sheet '{sheet}': {e}\n")

    except Exception as e:
        with open(error_log, "a", encoding="utf-8") as f:
            f.write(f"Could not read {filepath}: {e}\n")

    # --- progress ---
    percent = (idx / total_files) * 100
    print(f"Progress: {percent:.2f}% ({idx}/{total_files} files)")

# --- Step 3: save results ---
if all_student_numbers:
    df_out = pd.DataFrame(all_student_numbers, columns=["Student_No", "Source_File", "Sheet"])
    df_out.drop_duplicates(inplace=True)

    print("\nFirst 20 extracted numbers (debug):")
    print(df_out.head(20))  # check data

    df_out.to_excel(output_file, index=False, sheet_name="Student_Numbers", engine="openpyxl")
    print(f"\n✅ Extracted {len(df_out)} unique student numbers. Saved to {output_file}")
else:
    print("\n❌ No student numbers found.")


Found 1262 Excel files to process.
Progress: 0.08% (1/1262 files)
Progress: 0.16% (2/1262 files)
Progress: 0.24% (3/1262 files)
Progress: 0.32% (4/1262 files)
Progress: 0.40% (5/1262 files)
Progress: 0.48% (6/1262 files)
Progress: 0.55% (7/1262 files)
Progress: 0.63% (8/1262 files)
Progress: 0.71% (9/1262 files)
Progress: 0.79% (10/1262 files)
Progress: 0.87% (11/1262 files)
Progress: 0.95% (12/1262 files)
Progress: 1.03% (13/1262 files)
Progress: 1.11% (14/1262 files)
Progress: 1.19% (15/1262 files)
Progress: 1.27% (16/1262 files)
Progress: 1.35% (17/1262 files)
Progress: 1.43% (18/1262 files)
Progress: 1.51% (19/1262 files)
Progress: 1.58% (20/1262 files)
Progress: 1.66% (21/1262 files)
Progress: 1.74% (22/1262 files)
Progress: 1.82% (23/1262 files)
Progress: 1.90% (24/1262 files)
Progress: 1.98% (25/1262 files)
Progress: 2.06% (26/1262 files)
Progress: 2.14% (27/1262 files)
Progress: 2.22% (28/1262 files)
Progress: 2.30% (29/1262 files)
Progress: 2.38% (30/1262 files)
Progress: 2.46

In [2]:
#filtered result


import os
import pandas as pd
# file for results
# path to your folder
main_folder = r"YOUR_OUTPUT_FOLDER"
# path to your master student list
master_file = r"all student.xlsx"

# load student numbers
master_df = pd.read_excel(master_file)
student_numbers = set(master_df.iloc[:, 0].astype(str))  # first column

# prepare final results
all_matches = []

# walk through all subfolders
for root, dirs, files in os.walk(main_folder):
    for file in files:
        if file.endswith((".xlsx", ".xls")):
            file_path = os.path.join(root, file)
            try:
                df = pd.read_excel(file_path, dtype=str)  # read as text

                # try to find the MODULE CODE row once per file
                module_code_row = None
                for _, row in df.iterrows():
                    row_text = " ".join(str(x) for x in row.values if pd.notna(x))
                    if "CODE" in row_text.upper():
                        module_code_row = list(row.values)
                        break

                # now check each row for student numbers
                for _, row in df.iterrows():
                    row_text = " ".join(str(x) for x in row.values if pd.notna(x))
                    found_sn = None
                    for sn in student_numbers:
                        if sn in row_text:
                            found_sn = sn
                            break
                    if found_sn:
                        # save student row
                        row_data = list(row.values)
                        row_data.append(file_path)  # keep file source
                        all_matches.append(row_data)

                        # save module code row (if found)
                        if module_code_row:
                            all_matches.append(module_code_row)
            except Exception as e:
                print(f"Skipping {file_path}: {e}")

# save results
if all_matches:
    result_df = pd.DataFrame(all_matches)
    result_df.to_excel("filtered_results_all_centre.xlsx", index=False, header=False)
    print("✅ Finished! Results saved in filtered_results.xlsx")
else:
    print("⚠️ No matching student numbers found.")


Skipping YOUR_OUTPUT_FOLDER\1217_SEM -IV - Old (Chibombo)_BE.xlsx: Excel file format cannot be determined, you must specify an engine manually.
Skipping YOUR_OUTPUT_FOLDER\1221_SEM - I BSC CS B15_BSC.xlsx: "No such keys(s): 'io.excel.zip.reader'"
Skipping YOUR_OUTPUT_FOLDER\1222_SEM -I BE_BE.xlsx: Excel file format cannot be determined, you must specify an engine manually.
Skipping YOUR_OUTPUT_FOLDER\1291_SEM -VI spcl mar-25_BSC.xlsx: Excel file format cannot be determined, you must specify an engine manually.
Skipping YOUR_OUTPUT_FOLDER\1292_SEM I  FASTTRACK spcl mar-25_BSC.xlsx: Excel file format cannot be determined, you must specify an engine manually.
Skipping YOUR_OUTPUT_FOLDER\1322_BATCH 1 SEM 6_DIP(N).xlsx: Excel file format cannot be determined, you must specify an engine manually.
Skipping YOUR_OUTPUT_FOLDER\1692_B8 MBA F&IB IE FORM MAY 2025_MODULE 4.xlsx: File is not a zip file
✅ Finished! Results saved in filtered_results.xlsx


In [4]:
#cleaned result

import pandas as pd

# Load your Excel file
file_path = r"filtered_results_all_centre.xlsx"   # change this
df = pd.read_excel(file_path)

# ---- STEP 1: Separate student info and module data ----
# Adjust these columns based on your file
student_cols = df.iloc[:, :2]   # e.g., Student No, Name, etc.
module_data = df.iloc[:, 2:]    # rest are module columns

# ---- STEP 2: Define chunk size ----
chunk_size = 5  # module_code, mark1, mark2, total, grade

modules = []

expected_cols = ["module_code", "mark1", "mark2", "total", "grade"]

# ---- STEP 3: Loop through module columns ----
for i in range(0, module_data.shape[1], chunk_size):
    chunk = module_data.iloc[:, i:i+chunk_size].copy()
    
    # Skip incomplete chunks
    if chunk.shape[1] != chunk_size:
        print(f"Skipping chunk at index {i}, found {chunk.shape[1]} columns")
        continue
    
    # Rename columns
    chunk.columns = expected_cols
    
    # Combine with student info
    merged = pd.concat([student_cols, chunk], axis=1)
    
    # Remove rows where module_code is missing (optional cleanup)
    merged = merged[merged["module_code"].notna()]
    
    modules.append(merged)

# ---- STEP 4: Combine all modules ----
final_df = pd.concat(modules, ignore_index=True)
# ---- STEP 5: Save output ---- 
output_file = r"cleaned_result 2 folder.xlsx" 
final_df.to_excel(output_file, index=False) 

print("Processing complete. File saved as:", output_file)

# ---- STEP 5: Save output as CSV ----
#output_file = r"cleaned_MANSA_result.csv"
#final_df.to_csv(output_file, index=False)

#print("Processing complete. File saved as:", output_file)

Skipping chunk at index 75, found 1 columns
Processing complete. File saved as: cleaned_result 2 folder.xlsx


In [14]:
#formating filtered result
import xlwings as xw

FILE_PATH = r"filtered_results_all_centre - Copy (2).xlsx"   # <-- change this
SHEET_NAME = "Sheet1"           # <-- change if needed

ROWS = 5000

# Columns used in your macro (as Excel letters)
insert_cols = ["C", "H", "M", "R", "W", "AB", "AG", "AL", "AQ"]

def col_letter(col):
    return col

def run():
    app = xw.App(visible=False)
    wb = xw.Book(FILE_PATH)
    sht = wb.sheets[SHEET_NAME]

    # 1. Insert row 1
    sht.api.Rows(1).Insert()

    # 2. Delete column A
    sht.api.Columns("A").Delete()

    # 3. Process each column insertion + formula fill
    for col in insert_cols:
        sht.api.Columns(col).Insert()

        # put formula in row 2
        sht.range(f"{col}2").formula = f"=R[1]C[1]"

        # autofill down to row 2245
        sht.range(f"{col}2:{col}{ROWS}").formula = sht.range(f"{col}2").formula

    # 4. Convert all formulas to values (like Paste Special Values)
    used = sht.used_range
    used.value = used.value

    # 5. Apply filter (Column A blank)
    sht.api.Range(f"A1:BA{ROWS+2}").AutoFilter(Field=1, Criteria1="=")

    # 6. Delete visible rows except header
    last_row = sht.api.Cells(sht.api.Rows.Count, 1).End(-4162).Row

    for r in range(last_row, 2, -1):
        if sht.range(f"A{r}").value in (None, ""):
            sht.api.Rows(r).Delete()

    # remove filter
    sht.api.AutoFilterMode = False

    wb.save()
    wb.close()
    app.quit()

if __name__ == "__main__":
    run()

In [5]:
import os
import shutil

# 🔹 Change this to your main folder (where all folders are)
source_folder = r"C:\Users\USER\Documents\New folder (4)\New folder"

# 🔹 Where you want all Excel files to go
output_folder = r"main campus 2026"

# Create output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Counter to avoid file name conflicts
file_count = 0

# Walk through all folders and subfolders
for root, dirs, files in os.walk(source_folder):
    for file in files:
        if file.lower().endswith(".xlsx"):
            source_path = os.path.join(root, file)

            # Rename file to avoid overwriting same names
            new_name = f"{file_count}_{file}"
            destination_path = os.path.join(output_folder, new_name)

            shutil.copy2(source_path, destination_path)

            file_count += 1

print(f"Done! {file_count} Excel files copied to {output_folder}")

Done! 1811 Excel files copied to main campus 2026
